# 07 · Export data for the Tableau dashboard
Writes flat CSVs to `tableau/` so the dashboard needs no Python. Every file comes from the **held-out test set**, so all scores are out-of-sample.

| File | One row per | Used for |
|---|---|---|
| `scored_encounters.csv` | test encounter | call list, risk distribution, who gets called by group |
| `policy_curve.csv` | call-list size (0–100% in 0.5% steps) | net-benefit curve driven by Tableau parameters |
| `gains_by_decile.csv` | risk decile | readmission rate by decile |
| `sensitivity_grid.csv` | program cost × effect | best % to call heatmap |
| `fairness_by_group.csv`, `shap_importance.csv` | group / feature | copied from Step 5 |

The data is the public, de-identified UCI dataset, so these files are safe to commit and publish on Tableau Public.

In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

PROC, RES, OUT = Path("data/processed"), Path("results"), Path("tableau")
OUT.mkdir(exist_ok=True)

pred = pd.read_parquet(PROC / "test_predictions.parquet")
df = pd.read_parquet(PROC / "model_table.parquet")
test_patients = set(pd.read_parquet(PROC / "test_patients.parquet")["patient_nbr"])
test = df[df["patient_nbr"].isin(test_patients)].reset_index(drop=True)
assert len(test) == len(pred) and (test["patient_nbr"].values == pred["patient_nbr"].values).all()
test["p_risk"] = pred["p_risk"].values
N = len(test)
print(f"Test encounters: {N:,}")

Test encounters: 19,773


## 1. Scored encounters (one row per test discharge)

In [2]:
# Readable labels for discharge disposition (UCI IDs_mapping.csv)
DISPOSITION = {1: "Home", 2: "Short-term hospital", 3: "Skilled nursing facility", 4: "Intermediate care facility",
               5: "Other inpatient institution", 6: "Home with home health", 7: "Left against medical advice",
               8: "Home with IV provider", 9: "Admitted as inpatient", 10: "Neonate transfer",
               12: "Expected outpatient return", 15: "Swing bed", 16: "Other outpatient institution",
               17: "This institution outpatient", 18: "Not recorded", 22: "Rehab facility", 23: "Long-term care hospital",
               24: "Medicaid nursing facility", 25: "Not mapped", 26: "Unknown", 27: "Federal facility",
               28: "Psychiatric hospital", 29: "Critical access hospital", 30: "Other health care institution"}

out = test.sort_values("p_risk", ascending=False, kind="stable").reset_index(drop=True)
out.insert(0, "risk_rank", np.arange(1, N + 1))
out["rank_share"] = out["risk_rank"] / N                      # 0.0001 = riskiest, 1.0 = lowest risk
out["risk_decile"] = (np.arange(N) * 10 // N) + 1             # 1 = highest risk
out["readmitted_30"] = np.where(out["readmit_30"] == 1, "Readmitted", "Not readmitted")

if "age_num" in out:
    out["age_group"] = pd.cut(out["age_num"], [0, 40, 60, 80, 120], right=False,
                              labels=["Under 40", "40-59", "60-79", "80+"]).astype(str)
if "male" in out:
    out["sex"] = np.where(out["male"] == 1, "Male", "Female")
if "discharge_disposition_id" in out:
    ids = pd.to_numeric(out["discharge_disposition_id"].astype(str), errors="coerce")
    out["discharge_disposition"] = [DISPOSITION.get(int(i), f"ID {int(i)}") if pd.notna(i) else "Unknown" for i in ids]

# Drop log-transformed helper columns: Tableau users read the raw counts
out = out.drop(columns=[c for c in out.columns if c.startswith("log_")])
out.to_csv(OUT / "scored_encounters.csv", index=False)
print(f"scored_encounters.csv: {out.shape[0]:,} rows, {out.shape[1]} columns")
out.head(3)

scored_encounters.csv: 19,773 rows, 50 columns


,risk_rank,patient_nbr,race,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,med_change,on_diabetes_med,male,p_risk,rank_share,risk_decile,readmitted_30,age_group,sex,discharge_disposition
0,1,33379713,Caucasian,2,6,1,9,OG,Other,33,...,0,1,0,0.649621,0.000051,1,Readmitted,40-59,Female,Home with home health
1,2,14780889,Caucasian,3,2,4,8,Unknown,Other,26,...,0,0,1,0.613185,0.000101,1,Readmitted,40-59,Male,Short-term hospital
2,3,43140906,Caucasian,1,6,7,3,MC,Unknown,11,...,1,1,0,0.610911,0.000152,1,Readmitted,Under 40,Female,Home with home health


## 2. Policy curve
Cumulative counts only. Tableau computes net benefit from these with parameters for readmission cost, program cost and effect, so the curve updates live.

In [3]:
y_sorted = out["readmit_30"].to_numpy()
cum = np.concatenate([[0], np.cumsum(y_sorted)])
shares = np.round(np.arange(0, 100.5, 0.5), 1)
called = np.round(shares / 100 * N).astype(int)
curve = pd.DataFrame({"share_called_pct": shares, "called": called, "readmits_reached": cum[called],
                      "total_encounters": N, "total_readmits": int(y_sorted.sum()), "base_rate": y_sorted.mean()})
curve["recall"] = curve["readmits_reached"] / curve["total_readmits"]
curve["precision"] = np.where(curve["called"] > 0, curve["readmits_reached"] / curve["called"].clip(lower=1), np.nan)
curve.to_csv(OUT / "policy_curve.csv", index=False)
curve.iloc[[0, 10, 20, 40, 100, 200]]

,share_called_pct,called,readmits_reached,total_encounters,total_readmits,base_rate,recall,precision
0,0.0,0,0,19773,2244,0.113488,0.000000,NaN
10,5.0,989,333,19773,2244,0.113488,0.148396,0.336704
20,10.0,1977,565,19773,2244,0.113488,0.251783,0.285787
40,20.0,3955,909,19773,2244,0.113488,0.405080,0.229836
100,50.0,9886,1601,19773,2244,0.113488,0.713458,0.161946
200,100.0,19773,2244,19773,2244,0.113488,1.000000,0.113488


## 3. Gains by decile

In [4]:
gains = out.groupby("risk_decile").agg(encounters=("readmit_30", "size"), readmissions=("readmit_30", "sum"),
                                       readmit_rate=("readmit_30", "mean"), mean_predicted=("p_risk", "mean")).reset_index()
gains["share_of_all_readmits"] = gains["readmissions"] / gains["readmissions"].sum()
gains["cumulative_capture"] = gains["share_of_all_readmits"].cumsum()
gains.to_csv(OUT / "gains_by_decile.csv", index=False)
gains.round(3)

,risk_decile,encounters,readmissions,readmit_rate,mean_predicted,share_of_all_readmits,cumulative_capture
0,1,1978,565,0.286,0.284,0.252,0.252
1,2,1977,344,0.174,0.178,0.153,0.405
2,3,1977,279,0.141,0.141,0.124,0.529
3,4,1978,204,0.103,0.118,0.091,0.620
4,5,1977,209,0.106,0.101,0.093,0.713
5,6,1977,189,0.096,0.088,0.084,0.798
6,7,1978,145,0.073,0.076,0.065,0.862
7,8,1977,150,0.076,0.065,0.067,0.929
8,9,1977,89,0.045,0.054,0.040,0.969
9,10,1977,70,0.035,0.039,0.031,1.000


## 4. Sensitivity grid (readmission cost $15,000)

In [5]:
def best_policy(cost_readmit, cost_pp, effect):
    k = np.arange(1, N + 1)
    net = effect * np.cumsum(y_sorted) * cost_readmit - k * cost_pp
    i = int(net.argmax())
    return (k[i] / N, net[i] / N * 1000) if net[i] > 0 else (0.0, 0.0)

rows = []
for c in [50, 100, 200, 300, 500]:
    for e in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        s, nb_ = best_policy(15_000, c, e)
        rows.append({"cost_per_patient": c, "effect_pct": int(e * 100), "best_share_called": s, "net_per_1000": round(nb_)})
grid = pd.DataFrame(rows)
grid.to_csv(OUT / "sensitivity_grid.csv", index=False)
grid.pivot(index="cost_per_patient", columns="effect_pct", values="best_share_called").round(2)

effect_pct,5,10,15,20,30,40
cost_per_patient,,,,,,
50,0.78,0.99,1.00,1.00,1.00,1.00
100,0.26,0.78,0.84,0.99,1.00,1.00
200,0.06,0.26,0.61,0.78,0.84,0.99
300,0.01,0.11,0.26,0.52,0.78,0.84
500,0.00,0.01,0.07,0.19,0.32,0.64


## 5. Copy Step 5 results

In [6]:
for src, dst in [("05_fairness_by_group.csv", "fairness_by_group.csv"), ("05_shap_importance.csv", "shap_importance.csv")]:
    if (RES / src).exists():
        shutil.copy(RES / src, OUT / dst)
        print(f"Copied {dst}")
    else:
        print(f"Missing results/{src}: rerun notebook 05")

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:5.2f} MB")

Copied fairness_by_group.csv
Copied shap_importance.csv
fairness_by_group.csv         0.00 MB
gains_by_decile.csv           0.00 MB
policy_curve.csv              0.02 MB
scored_encounters.csv         5.08 MB
sensitivity_grid.csv          0.00 MB
shap_importance.csv           0.00 MB


## 6. Data file for the Streamlit app
The test encounters with their features and risk scores, keeping column types (parquet), so `app.py` runs on Streamlit Community Cloud without the git-ignored `data/` folder.

In [7]:
import pandas as pd
from pathlib import Path

PROC_DIR = Path("data/processed")
APP_OUT = Path("tableau/app_scored.parquet")

model_table = pd.read_parquet(PROC_DIR / "model_table.parquet")
test_pred = pd.read_parquet(PROC_DIR / "test_predictions.parquet")
test_patients = set(pd.read_parquet(PROC_DIR / "test_patients.parquet")["patient_nbr"])

app_df = model_table[model_table["patient_nbr"].isin(test_patients)].reset_index(drop=True)
assert len(app_df) == len(test_pred), f"row mismatch: {len(app_df)} vs {len(test_pred)}"
app_df["p_risk"] = test_pred["p_risk"].values

APP_OUT.parent.mkdir(exist_ok=True)
app_df.to_parquet(APP_OUT, index=False)
print(f"app_scored.parquet: {len(app_df):,} rows, {APP_OUT.stat().st_size / 1e6:.2f} MB")

app_scored.parquet: 19,773 rows, 0.61 MB
